In [ ]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import sys, re, inspect
from pathlib import Path
import pandas as pd

HOME = Path.home()
CLIPS_DIR   = KWF_TEST_DIR / "split_clips"
OUTPUTS_DIR = KWF_TEST_DIR / "outputs"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(PIPELINE_DIR / "scripts"))

try:
    from metadata_extraction import extract_metadata
    sig = inspect.signature(extract_metadata)
    print("Signature:", sig)
    print("Master-script args NOT accepted:", {"input_folder", "output_csv"} - set(sig.parameters) or "none")
except Exception as e:
    print("IMPORT/SIGNATURE PROBLEM:", type(e).__name__, e)

IMPORT/SIGNATURE PROBLEM: ModuleNotFoundError No module named 'pydub'


What this does: sets up paths and imports extract_metadata, then compares its parameter names with the ones run_pipeline.py passes (input_folder, output_csv).

How to read the output: “none” means the master script’s Step 5 call is compatible. Any name listed would make run_pipeline.py crash at Step 5. An import error means the script can’t load at all, and the exact error is the finding.

In [4]:
import importlib.metadata as md
for pkg in ["pydub", "mutagen"]:
    try:
        print(f"{pkg:10s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:10s} NOT INSTALLED")

pydub      NOT INSTALLED
mutagen    NOT INSTALLED


In [5]:
%pip install pydub mutagen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mutagen]m1/2 [mutagen]
Note: you may need to restart the kernel to use updated packages.


In [6]:
from metadata_extraction import extract_metadata
sig = inspect.signature(extract_metadata)
print("Signature:", sig)
print("Master-script args NOT accepted:", {"input_folder", "output_csv"} - set(sig.parameters) or "none")

Signature: (input_folder, output_csv)
Master-script args NOT accepted: none


## Prediction for Step 5 (from reading the script, before running)
- Input: 1,747 clips in split_clips/. Expected output: 1,747 rows.
- Columns: clip_name, Recorder, Timestamp, Duration (sec), Sample Rate (Hz), Channels, Bit Depth, File Size (KB), Battery Voltage, Gain, Spectral Centroid, Zero Crossing Rate.
- clip_name keeps ".wav" (matches Steps 2 and 3, not Step 4).
- Recorder = "Audio_Moth_5". Timestamp is a text string like "20250319_163237" (local time, no timezone, not a datetime).
- Duration 3.0, Sample Rate 48000, Channels 1, Bit Depth 16, File Size about 281 KB.
- Battery Voltage and Gain: expected "Unknown" for every clip. The splitter rewrites each clip with soundfile, which drops the original AudioMoth header, and the script only looks for ID3-style tags.
- The script lists files with os.listdir (not recursive). Steps 2 to 4 use rglob, so a nested clips folder would give 0 rows here.
- Only files ending in lowercase ".wav" are read.
- Spectral Centroid and Zero Crossing Rate are extra librosa features, separate from Step 4's 10.
- Failed clips are skipped (printed, not in the CSV), so rows could be fewer than 1,747; "Error reading" lines are the only evidence of failures.
- Row order follows os.listdir (unsorted).
- extract_metadata returns None and only writes the CSV; it does not create the output folder.
- An input folder with no top-level .wav files produces an empty CSV with no error.

In [ ]:
import time

meta_csv = OUTPUTS_DIR / "nb05_metadata.csv"

t0 = time.time()

extract_metadata(
    input_folder=str(CLIPS_DIR),
    output_csv=str(meta_csv),
)

print(f"Runtime: {time.time() - t0:.1f}s")

.
Metadata saved to /Users/huyguy/kwf_test/outputs/nb05_metadata.csv
Runtime: 25.4s


What this does: runs extract_metadata on the 1,747 recorder 5 clips with the same arguments run_pipeline.py uses in step5_metadata, and times it.

How to read the output: a dot is printed every 1,000 clips, so one is expected. Any Error reading ... line is a clip that failed and will be missing from the CSV. “Metadata saved to ...” means it finished. The runtime is for the record. 

In [9]:
m = pd.read_csv(meta_csv)
print("Shape:", m.shape)
print("Columns:", m.columns.tolist())
print("Unique clip_names:", m["clip_name"].nunique(), "| nulls:", int(m.isna().sum().sum()))

# Name agreement with the other steps
on_disk = {p.name for p in CLIPS_DIR.glob("*.wav")}
cnn = pd.read_csv(OUTPUTS_DIR / "nb02_recorder5_rerun_predictions.csv")
bird = pd.read_csv(OUTPUTS_DIR / "nb03_birdnet_output.csv")
ac = pd.read_csv(OUTPUTS_DIR / "nb04_recorder5_acoustic_features.csv")
names = set(m["clip_name"])
print("\nMatch with Step 1 files:", names == on_disk)
print("Match with Step 2:", names == set(cnn["clip_name"]), "| Step 3:", names == set(bird["clip_name"]))
print("Exact match with Step 4:", len(names & set(ac["clip_name"])),
      "| after adding .wav to Step 4:", len(names & {n + ".wav" for n in ac["clip_name"]}))

# Values
print("\nRecorder:", m["Recorder"].unique().tolist())
for col in ["Duration (sec)", "Sample Rate (Hz)", "Channels", "Bit Depth"]:
    print(f"{col}:", m[col].value_counts().to_dict())
print("File Size (KB):", m["File Size (KB)"].min(), "to", m["File Size (KB)"].max())
print("Battery Voltage:", m["Battery Voltage"].value_counts().to_dict())
print("Gain:", m["Gain"].value_counts().to_dict())

# Timestamp and order
print("\nExample Timestamp:", repr(m["Timestamp"].iloc[0]), "| dtype:", m["Timestamp"].dtype)
ts = pd.to_datetime(m["Timestamp"].astype(str), format="%Y%m%d_%H%M%S")
print("Time span:", ts.min(), "->", ts.max())
print("Gaps after sorting (s):", ts.sort_values().diff().dropna().dt.total_seconds().value_counts().to_dict())
print("Rows sorted by clip_name:", m["clip_name"].is_monotonic_increasing)

Shape: (1747, 12)
Columns: ['clip_name', 'Recorder', 'Timestamp', 'Duration (sec)', 'Sample Rate (Hz)', 'Channels', 'Bit Depth', 'File Size (KB)', 'Battery Voltage', 'Gain', 'Spectral Centroid', 'Zero Crossing Rate']
Unique clip_names: 1747 | nulls: 0

Match with Step 1 files: True
Match with Step 2: True | Step 3: True
Exact match with Step 4: 0 | after adding .wav to Step 4: 1747

Recorder: ['Audio_Moth_5']
Duration (sec): {3.0: 1747}
Sample Rate (Hz): {48000: 1747}
Channels: {1: 1747}
Bit Depth: {16: 1747}
File Size (KB): 281.29 to 281.29
Battery Voltage: {'Unknown': 1747}
Gain: {'Unknown': 1747}

Example Timestamp: '20250319_174516' | dtype: str
Time span: 2025-03-19 16:32:37 -> 2025-03-19 17:59:55
Gaps after sorting (s): {3.0: 1746}
Rows sorted by clip_name: False


What this does: loads the CSV and checks it against the prediction: row count, columns, name matches with Steps 1 to 4, the values of the audio properties, the AudioMoth tags (Battery Voltage and Gain), the Timestamp format, and row order.

How to read the output: 1,747 rows and 12 columns means no clip was dropped. Name matches show which steps agree. Agree with step 1 to 3 and with 4 once ".wav" is added. Battery Voltage and Gain are all “Unknown”, the splitter removed the original AudioMoth header. “Sorted: False” confirms the unsorted order. The Timestamp check shows  the text parses into real times with exact 3-second gaps.

In [12]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
RAW_DIR = KWF_TEST_DIR / "raw_audio"

print(m[["Spectral Centroid", "Zero Crossing Rate"]].describe().T[["min", "mean", "max"]])

raw = RAW_DIR / "Audio_Moth_5_031925" / "20250319_163237.WAV"
clip = next(CLIPS_DIR.glob("*.wav"))
for label, f in [("raw file", raw), ("split clip", clip)]:
    head = open(f, "rb").read(8192)
    print(f"\n{label}: 'AudioMoth' in header: {b'AudioMoth' in head} | 'battery' in header: {b'battery' in head.lower()}")

                            min         mean           max
Spectral Centroid   5469.642762  9036.051165  12830.058548
Zero Crossing Rate     0.061757     0.323229      0.555051

raw file: 'AudioMoth' in header: True | 'battery' in header: True

split clip: 'AudioMoth' in header: False | 'battery' in header: False


What this does: summarizes the two librosa features Step 5 adds (Spectral Centroid, Zero Crossing Rate), and searches the raw recording and one split clip for AudioMoth settings text in the file header.

How to read the output: the feature ranges should be plausible and nonzero. For the header check, “True” means the text is in that file. If the raw file has it and the split clip doesn’t, the splitter drops the header. If neither has it, the script’s tag lookup is the likelier reason for “Unknown”.

In [ ]:
import shutil, tempfile

clips = sorted(CLIPS_DIR.glob("*.wav"))[:10]
test_root = Path(tempfile.mkdtemp(prefix="nested_test_"))

# Mixed: 5 clips at the top level, 5 inside a subfolder
mixed = test_root / "mixed"
(mixed / "sub").mkdir(parents=True)
for c in clips[:5]:
    shutil.copy(c, mixed / c.name)
for c in clips[5:]:
    shutil.copy(c, mixed / "sub" / c.name)

# Nested only: all 10 clips inside a subfolder
nested = test_root / "nested_only"
(nested / "sub").mkdir(parents=True)
for c in clips:
    shutil.copy(c, nested / "sub" / c.name)

for label, folder in [("mixed (5 top + 5 nested)", mixed), ("nested only (10)", nested)]:
    out = test_root / f"{folder.name}.csv"
    extract_metadata(input_folder=str(folder), output_csv=str(out))
    try:
        rows = len(pd.read_csv(out))
        print(f"{label}: Rows: {rows}")
    except pd.errors.EmptyDataError:
        print(f"{label}: Empty CSV (EmptyDataError)")
    print(f"   clips a recursive search finds: {len(list(folder.rglob('*.wav')))}\n")

shutil.rmtree(test_root)


Metadata saved to /var/folders/18/gw6v9vc95bg8w0jxzmjwm2br0000gn/T/nested_test_w23h5v5t/mixed.csv
mixed (5 top + 5 nested): Rows: 5
   clips a recursive search finds: 10


Metadata saved to /var/folders/18/gw6v9vc95bg8w0jxzmjwm2br0000gn/T/nested_test_w23h5v5t/nested_only.csv
nested only (10): Empty CSV (EmptyDataError)
   clips a recursive search finds: 10



What this does: builds the small test folder, runs extract_metadata on it, then runs it on a folder containing only a subfolder. It counts the rows each run produced.

How to read the output: “Rows: 5” for the mixed folder means the nested clips were silently skipped, with no error. “Empty CSV” for the nested-only folder means a nested layout would produce no metadata at all.

In [ ]:
test_root = Path(tempfile.mkdtemp(prefix="upper_test_"))

for c in sorted(CLIPS_DIR.glob("*.wav"))[:3]:
    shutil.copy(c, test_root / (c.stem + ".WAV"))

print("Files in folder:", sorted(p.name for p in test_root.iterdir()))
print("Names ending in lowercase .wav:", sum(p.name.endswith(".wav") for p in test_root.iterdir()))
print("Names ending in uppercase .WAV:", sum(p.name.endswith(".WAV") for p in test_root.iterdir()))

out = test_root / "upper.csv"
extract_metadata(input_folder=str(test_root), output_csv=str(out))

try:
    print("Rows:", len(pd.read_csv(out)))
except pd.errors.EmptyDataError:
    print("Empty CSV (EmptyDataError)")

shutil.rmtree(test_root)

Files in folder: ['Audio_Moth_5_20250319_163237.WAV', 'Audio_Moth_5_20250319_163240.WAV', 'Audio_Moth_5_20250319_163243.WAV']
Names ending in lowercase .wav: 0
Names ending in uppercase .WAV: 3

Metadata saved to /var/folders/18/gw6v9vc95bg8w0jxzmjwm2br0000gn/T/upper_test_00l1i4ju/upper.csv
Empty CSV (EmptyDataError)


What this does: copies 3 clips into a temporary folder with their extension changed to .WAV, then runs extract_metadata on it and counts the rows.

How to read the output: “Empty CSV” confirms uppercase .WAV files are silently ignored. “Rows: 3” would mean my reading of the code was wrong. The last two lines show how many files each spelling matches in a case-sensitive search, so we know the files really have the uppercase extension.

## Observations (Step 5)

**Tested:** `extract_metadata` on the 1,747 recorder 5 clips, using the same call as `run_pipeline.py`.

**Result:** works. 1,747 clips in, 1,747 rows out (12 columns, no missing values). File names match Steps 1, 2 and 3. Every clip is 3.0 s, 48 kHz, mono, 16-bit, and the timestamps run from 16:32:37 to 17:59:55 in exact 3-second steps.

**Issues**
1. `pydub` and `mutagen` were not installed and are not in the documented dependencies, so the pipeline would crash on a fresh machine. I installed them for this test.
2. Battery Voltage and Gain are "Unknown" for every clip. The original recording has this information in its header, but Step 1 does not copy the header into the clips.
3. Rows are not in time order, so later steps that compare neighboring clips must sort first.
4. Step 4's `clip_name` has no ".wav", unlike Steps 2, 3 and 5.
5. Nested folders: `extract_metadata` reads only the top-level folder. In a test with 5 top-level and 5 nested clips it returned 5 rows with no warning; with only nested clips it wrote an empty CSV, yet still printed "Metadata saved". Steps 2 to 4 read nested folders, so the steps disagree on layout. Low risk in the current flow (Step 1 writes a flat folder), but it would silently lose data on nested input.

**Consideration:** does Step 7 or the HP model use Battery Voltage or Gain?

**Robustness note (low risk):** `extract_metadata` only reads names ending in lowercase ".wav". Three clips renamed to ".WAV" produced an empty CSV, yet "Metadata saved" was still printed. This does not affect the pipeline as run, because Step 1 always writes lowercase ".wav", but Step 5 would silently skip raw AudioMoth files (named ".WAV") if used on them directly.